# Narrower experts, more experts — dense GPT → sparse MoE

Self-contained notebook for Google Colab, intended for a **T4 / 16 GB**. Completed T4 results and evidence are documented in README.md. Select **Runtime → Change runtime type → GPU**, then run the cells in order.

We train our own causal GPT on a bounded TinyStories subset with a train-only BPE tokenizer. All continuation arms use the same dense checkpoint, batches, token budget and a reset AdamW optimizer.

| Arm | FFN width per expert | Stored experts | Selected experts/token | Selected FFN width |
|---|---:|---:|---:|---:|
| Dense continuation | 384 | 1 | 1 | 384 |
| Full-size copy | 384 | 4 | 1 | 384 |
| Partitioned copy | 96 | 16 | 4 | 384 |

Both MoEs store four times the dense FFN weights, plus routers. This matches **FFN matrix MACs**, not whole-model runtime: router size, top-k, dispatch and optimizer work differ. The narrow experts copy matching gate/up rows and down columns; they are not a learned compression of the entire FFN.

The dense FFN is the sum of four partitions. Normalized routing averages its selected outputs, so we multiply the partitioned mixture by four. Uniformly selecting one of each partition would preserve the original output. Flat top-4 can omit partitions or select their replicas, and therefore can change predictions immediately. We measure that loss jump and its recovery.


In [ ]:
import sys, subprocess, torch
assert torch.cuda.is_available(), "Enable a Colab GPU runtime before continuing."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", torch.cuda.get_device_properties(0).total_memory / 2**30)
print("PyTorch:", torch.__version__)
subprocess.run([sys.executable, "-m", "pip", "install", "datasets>=3", "tokenizers>=0.20", "huggingface_hub", "numpy", "matplotlib"], check=True)


In [ ]:
import json, os
from pathlib import Path
root = Path("/content/MoE_Granular_Lab")
root.mkdir(exist_ok=True)
os.chdir(root)
# Embedded source snapshot: no GitHub repository or external checkout needed.
payload = {'src/gpt.py': '"""Small causal decoder-only Transformer and FFN-upcycled sparse MoE."""\nimport copy\nimport math\nfrom dataclasses import dataclass\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n@dataclass\nclass GPTConfig:\n    vocab_size: int = 65\n    context: int = 128\n    layers: int = 4\n    width: int = 128\n    heads: int = 4\n    intermediate: int = 256\n    experts: int = 4\n    top_k: int = 2\n\nclass SwiGLU(nn.Module):\n    def __init__(self, width, intermediate):\n        super().__init__()\n        self.gate = nn.Linear(width, intermediate, bias=False)\n        self.up = nn.Linear(width, intermediate, bias=False)\n        self.down = nn.Linear(intermediate, width, bias=False)\n\n    def forward(self, x):\n        return self.down(F.silu(self.gate(x)) * self.up(x))\n\nclass ExpertFFN(nn.Module):\n    def __init__(self, dense_ffn, width, experts, top_k):\n        super().__init__()\n        if not 1 <= top_k <= experts:\n            raise ValueError(\'Require 1 <= top_k <= experts\')\n        self.top_k = top_k\n        self.experts = nn.ModuleList([copy.deepcopy(dense_ffn) for _ in range(experts)])\n        self.router = nn.Linear(width, experts, bias=False)\n        nn.init.normal_(self.router.weight, std=.02)\n\n    def forward(self, hidden):\n        shape = hidden.shape\n        x = hidden.reshape(-1, shape[-1])\n        # FP32 for ordinary training/autocast, FP64 for independent CPU\n        # reference validation. Do not downcast a double reference gate.\n        gate_dtype = torch.float64 if x.dtype == torch.float64 else torch.float32\n        logits = F.linear(x.to(gate_dtype), self.router.weight.to(gate_dtype))\n        probs = logits.softmax(-1)\n        values, ids = logits.topk(self.top_k, dim=-1)\n        weights = values.softmax(-1).to(x.dtype)\n        output = torch.zeros_like(x)\n        for i, expert in enumerate(self.experts):\n            rows, slots = torch.where(ids == i)\n            if rows.numel():\n                contribution = expert(x[rows]) * weights[rows, slots, None]\n                output = output.index_add(0, rows, contribution)\n        counts = torch.bincount(ids.flatten(), minlength=len(self.experts)).float()\n        fraction = counts / ids.numel()\n        balance = len(self.experts) * (fraction.detach() * probs.mean(0)).sum()\n        entropy = -(probs * probs.clamp_min(1e-9).log()).sum(-1).mean()\n        return output.reshape(shape), {\'balance\': balance, \'counts\': counts, \'entropy\': entropy}\n\nclass Attention(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        if cfg.width % cfg.heads:\n            raise ValueError(\'width must be divisible by heads\')\n        self.heads = cfg.heads\n        self.qkv = nn.Linear(cfg.width, 3*cfg.width, bias=False)\n        self.proj = nn.Linear(cfg.width, cfg.width, bias=False)\n\n    def forward(self, x):\n        b,t,d = x.shape\n        q,k,v = self.qkv(x).chunk(3, dim=-1)\n        q,k,v = [a.reshape(b,t,self.heads,d//self.heads).transpose(1,2) for a in (q,k,v)]\n        y = F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.)\n        return self.proj(y.transpose(1,2).contiguous().reshape(b,t,d))\n\nclass Block(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.norm_attn = nn.LayerNorm(cfg.width)\n        self.attn = Attention(cfg)\n        self.norm_ffn = nn.LayerNorm(cfg.width)\n        self.ffn = SwiGLU(cfg.width, cfg.intermediate)\n\n    def forward(self, x):\n        x = x + self.attn(self.norm_attn(x))\n        if isinstance(self.ffn, ExpertFFN):\n            y, aux = self.ffn(self.norm_ffn(x))\n        else:\n            y, aux = self.ffn(self.norm_ffn(x)), None\n        return x+y, aux\n\nclass TinyGPT(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = cfg\n        self.token_embedding = nn.Embedding(cfg.vocab_size,cfg.width)\n        self.position_embedding = nn.Embedding(cfg.context,cfg.width)\n        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.layers)])\n        self.norm = nn.LayerNorm(cfg.width)\n        self.head = nn.Linear(cfg.width,cfg.vocab_size,bias=False)\n        self.head.weight = self.token_embedding.weight\n        self.apply(self._initialize)\n        for name,p in self.named_parameters():\n            if name.endswith((\'attn.proj.weight\',\'ffn.down.weight\')):\n                nn.init.normal_(p, std=.02/math.sqrt(2*cfg.layers))\n\n    @staticmethod\n    def _initialize(module):\n        if isinstance(module,(nn.Linear,nn.Embedding)):\n            nn.init.normal_(module.weight,std=.02)\n            if isinstance(module,nn.Linear) and module.bias is not None:\n                nn.init.zeros_(module.bias)\n\n    def forward(self, tokens):\n        if tokens.shape[1] > self.cfg.context:\n            raise ValueError(\'Sequence exceeds configured context\')\n        x = self.token_embedding(tokens) + self.position_embedding(torch.arange(tokens.shape[1],device=tokens.device))\n        aux=[]\n        for block in self.blocks:\n            x,a = block(x)\n            if a is not None:\n                aux.append(a)\n        return self.head(self.norm(x)), aux\n\n    def upcycle(self):\n        result = copy.deepcopy(self)\n        if any(isinstance(b.ffn,ExpertFFN) for b in result.blocks):\n            raise ValueError(\'Model is already an MoE\')\n        for block in result.blocks:\n            block.ffn = ExpertFFN(block.ffn,self.cfg.width,self.cfg.experts,self.cfg.top_k).to(self.head.weight.device)\n        return result\n\n    def parameter_counts(self):\n        total=sum(p.numel() for p in self.parameters())\n        inactive=0\n        for block in self.blocks:\n            if isinstance(block.ffn,ExpertFFN):\n                each=sum(p.numel() for p in block.ffn.experts[0].parameters())\n                inactive += (len(block.ffn.experts)-block.ffn.top_k)*each\n        return {\'total\':total, \'active_parameter_convention\':total-inactive,\n                \'convention\':\'All nonexpert parameters, including full tied embedding table, plus selected expert parameters. Not a FLOP count.\'}\n', 'src/granular.py': '"""Partition a trained SwiGLU into narrower, replicated sparse experts."""\nimport copy\nimport torch\nfrom torch import nn\nfrom .gpt import ExpertFFN, SwiGLU\n\n\nclass GranularFFN(ExpertFFN):\n    def __init__(self, dense, partitions, replicas, top_k):\n        nn.Module.__init__(self)\n        width = dense.gate.in_features\n        intermediate = dense.gate.out_features\n        if partitions < 1 or replicas < 1 or intermediate % partitions:\n            raise ValueError(\'Positive partitions/replicas required; partitions must divide FFN width\')\n        count = partitions * replicas\n        if not 1 <= top_k <= count:\n            raise ValueError(\'Require 1 <= top_k <= expert count\')\n        self.top_k = top_k\n        self.partitions = partitions\n        self.replicas = replicas\n        # Normalized routing weights average experts. Scale by partition count:\n        # uniform weights over one of each partition then reconstruct the sum.\n        self.output_scale = partitions\n        small = intermediate // partitions\n        self.experts = nn.ModuleList()\n        self.unit_indices = []\n        for replica in range(replicas):\n            for part in range(partitions):\n                indices = torch.arange(part * small, (part + 1) * small, device=dense.gate.weight.device)\n                expert = SwiGLU(width, small).to(device=dense.gate.weight.device, dtype=dense.gate.weight.dtype)\n                with torch.no_grad():\n                    expert.gate.weight.copy_(dense.gate.weight[indices])\n                    expert.up.weight.copy_(dense.up.weight[indices])\n                    expert.down.weight.copy_(dense.down.weight[:, indices])\n                self.experts.append(expert)\n                self.unit_indices.append(indices.cpu().tolist())\n        self.router = nn.Linear(width, count, bias=False).to(dense.gate.weight.device)\n        nn.init.normal_(self.router.weight, std=.02)\n\n    def forward(self, hidden):\n        output, aux = super().forward(hidden)\n        return output * self.output_scale, aux\n\n\ndef partition_upcycle(model, partitions, replicas, top_k):\n    if any(isinstance(b.ffn, ExpertFFN) for b in model.blocks):\n        raise ValueError(\'Partition conversion requires a dense GPT\')\n    result = copy.deepcopy(model)\n    for block in result.blocks:\n        block.ffn = GranularFFN(block.ffn, partitions, replicas, top_k)\n    result.cfg.intermediate //= partitions\n    result.cfg.experts = partitions * replicas\n    result.cfg.top_k = top_k\n    return result\n', 'src/granular_experiment.py': '"""Matched-token continuation: dense, full-copy MoE, and narrower partition MoE."""\nimport argparse\nimport copy\nimport hashlib\nimport json\nimport platform\nfrom dataclasses import asdict\nfrom pathlib import Path\nimport torch\nfrom .gpt import GPTConfig, TinyGPT\nfrom .granular import partition_upcycle\nfrom .train import load_data\nfrom .train_gpt import sample, evaluate, phase\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--config\', default=\'configs/granular_t4.json\')\n    parser.add_argument(\'--data\', default=\'data/tinystories_lab.npz\')\n    parser.add_argument(\'--device\', default=\'cpu\')\n    parser.add_argument(\'--output\', required=True)\n    args = parser.parse_args()\n    torch.set_num_threads(2)\n    cfg = json.loads(Path(args.config).read_text())\n    out = Path(args.output)\n    out.mkdir(parents=True, exist_ok=False)\n    (out / \'config.json\').write_text(json.dumps(cfg, indent=2))\n    data_path = Path(args.data)\n    if data_path.suffix == \'.npz\':\n        import numpy as np\n        with np.load(data_path) as data:\n            splits = {key: torch.from_numpy(data[key].astype(\'int64\')) for key in (\'train\', \'validation\', \'test\')}\n            vocab_size = int(data[\'vocab_size\'])\n        manifest_path = data_path.parent / \'tinystories_lab\' / \'manifest.json\'\n        manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}\n    else:\n        splits, manifest = load_data(data_path, cfg[\'model\'][\'context\'])\n        vocab_size = len(manifest[\'vocabulary\'])\n    manifest.update(token_array_sha256=hashlib.sha256(data_path.read_bytes()).hexdigest(),\n                    token_counts={k: len(v) for k, v in splits.items()})\n    (out / \'data_manifest.json\').write_text(json.dumps(manifest, indent=2))\n    (out / \'environment.json\').write_text(json.dumps({\n        \'python\': platform.python_version(), \'torch\': str(torch.__version__), \'device\': args.device,\n        \'gpu\': torch.cuda.get_device_name() if args.device.startswith(\'cuda\') else None,\n        \'precision\': \'FP32\', \'source_sha256\': {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in Path(\'src\').glob(\'*.py\')}\n    }, indent=2))\n    evals = {k: sample(v, torch.Generator().manual_seed(9000 + i), cfg[\'eval_sequences\'], cfg[\'model\'][\'context\'])\n             for i, (k, v) in enumerate(splits.items())}\n    summaries = []\n    for seed in cfg[\'seeds\']:\n        torch.manual_seed(seed)\n        checkpoint_dir = Path(\'checkpoints\') / out.name / str(seed)\n        checkpoint_dir.mkdir(parents=True, exist_ok=False)\n        dense = TinyGPT(GPTConfig(vocab_size=vocab_size, **cfg[\'model\'])).to(args.device)\n        records, timing = phase(dense, cfg, splits[\'train\'], evals, seed, \'dense_pretrain\', cfg[\'dense_steps\'],\n                                cfg[\'dense_lr\'], 0, args.device, checkpoint_dir, out / \'metrics.jsonl\')\n        # Keep only one branch on the GPU during continuation. All branches start\n        # from this same CPU dense checkpoint; all Adam states reset identically.\n        dense = dense.cpu().eval()\n        branches = {}\n        for name in (\'dense_control\', \'full_copy\', \'granular\'):\n            torch.manual_seed(seed + 30000)\n            if name == \'dense_control\':\n                branch = copy.deepcopy(dense)\n            elif name == \'full_copy\':\n                branch = dense.upcycle()\n            else:\n                branch = partition_upcycle(dense, **cfg[\'conversion\'])\n            branch = branch.to(args.device).eval()\n            original = {k: p.detach().cpu().clone() for k, p in branch.named_parameters() if \'.experts.\' in k}\n            with torch.no_grad():\n                probe = evals[\'validation\'][0]\n                before = dense(probe)[0]\n                after = branch(probe.to(args.device))[0].cpu()\n                drift = (before - after).abs().max().item()\n            if args.device.startswith(\'cuda\'):\n                torch.cuda.reset_peak_memory_stats()\n            continuation, branch_timing = phase(branch, cfg, splits[\'train\'], evals, seed, name,\n                cfg[\'continuation_steps\'], cfg[\'continuation_lr\'], cfg[\'dense_steps\'], args.device,\n                checkpoint_dir, out / \'metrics.jsonl\')\n            changes = {k: (p.detach().cpu() - original[k]).norm().item()\n                       for k, p in branch.named_parameters() if k in original}\n            h = branch.cfg.intermediate\n            k = branch.cfg.top_k if name != \'dense_control\' else 1\n            branches[name] = {\n                \'model_config\': asdict(branch.cfg), \'parameters\': branch.parameter_counts(),\n                \'selected_ffn_width\': h * k,\n                \'ffn_matrix_macs_per_token_per_layer\': 3 * branch.cfg.width * h * k,\n                \'router_matrix_macs_per_token_per_layer\': branch.cfg.width * branch.cfg.experts if original else 0,\n                \'conversion_max_logit_drift\': drift,\n                \'conversion_ce_jump\': continuation[0][\'validation_ce\'] - records[-1][\'validation_ce\'],\n                \'start\': continuation[0], \'final\': continuation[-1],\n                \'validation_drops_from_conversion\': continuation[-1][\'validation_ce\'] < continuation[0][\'validation_ce\'],\n                \'validation_recovers_dense_checkpoint\': continuation[-1][\'validation_ce\'] < records[-1][\'validation_ce\'],\n                \'expert_weight_changes_l2\': changes, \'all_expert_matrices_updated\': all(v > 0 for v in changes.values()) if changes else None,\n                \'test_ce\': evaluate(branch, evals[\'test\'], args.device)[\'task_ce\'],\n                \'peak_cuda_allocated_bytes\': torch.cuda.max_memory_allocated() if args.device.startswith(\'cuda\') else None,\n                **branch_timing}\n            del branch\n            if args.device.startswith(\'cuda\'):\n                torch.cuda.empty_cache()\n        summaries.append({\'seed\': seed, \'dense_initial\': records[0], \'dense_end\': records[-1],\n                          \'dense_timing\': timing, \'branches\': branches})\n        (out / \'summary.json\').write_text(json.dumps(summaries, indent=2))\n    lines = [\'# Narrow-expert continuation results\', \'\',\n             \'Measured run. Matched tokens/data order and reset Adam; FFN matrix MACs exclude attention, activation, routing selection and dispatch overhead.\', \'\',\n             \'| Seed | Arm | Total / active parameters | Conversion CE | Final validation CE | Loss drops / recovers dense |\',\n             \'|---|---|---|---|---|---|\']\n    for result in summaries:\n        for name, branch in result[\'branches\'].items():\n            p = branch[\'parameters\']\n            lines.append(f"| {result[\'seed\']} | {name} | {p[\'total\']} / {p[\'active_parameter_convention\']} | {branch[\'start\'][\'validation_ce\']:.6f} | {branch[\'final\'][\'validation_ce\']:.6f} | {branch[\'validation_drops_from_conversion\']} / {branch[\'validation_recovers_dense_checkpoint\']} |")\n    lines += [\'\', \'See summary.json for conversion drift, load fractions, gradient checks, checkpoint recovery, timings and parameter changes. No statistical quality or GPU efficiency claim follows from a one-seed CPU pilot.\']\n    (out / \'REPORT.md\').write_text(\'\\n\'.join(lines) + \'\\n\')\n\n\nif __name__ == \'__main__\':\n    main()\n', 'src/train_gpt.py': '"""Dense GPT -> FFN-upcycled MoE, with matched-token dense continuation."""\nimport argparse\nimport copy\nimport hashlib\nimport json\nimport platform\nimport subprocess\nimport time\nfrom dataclasses import asdict\nfrom pathlib import Path\nimport torch\nfrom torch.nn import functional as F\nfrom .gpt import GPTConfig,TinyGPT,ExpertFFN\nfrom .train import load_data\nfrom .conversion_checks import verify_model_conversion\n\n\ndef sample(data,gen,n,context):\n    starts=torch.randint(len(data)-context,(n,),generator=gen)\n    sequence=data[starts[:,None]+torch.arange(context+1)]\n    return sequence[:,:-1],sequence[:,1:]\n\n\ndef step(model,opt,batch,device,cfg):\n    model.train()\n    opt.zero_grad(set_to_none=True)\n    x,y=(t.to(device) for t in batch)\n    logits,aux=model(x)\n    ce=F.cross_entropy(logits.reshape(-1,logits.shape[-1]),y.reshape(-1))\n    balance=torch.stack([a[\'balance\'] for a in aux]).mean() if aux else ce.new_tensor(0.)\n    objective=ce+cfg[\'balance_coefficient\']*balance\n    objective.backward()\n    norm=torch.nn.utils.clip_grad_norm_(model.parameters(),cfg[\'gradient_clip\'])\n    if not torch.isfinite(norm) or not torch.isfinite(objective):\n        raise RuntimeError(\'Non-finite gradients/objective\')\n    router_grad=[b.ffn.router.weight.grad.norm().item() for b in model.blocks if isinstance(b.ffn,ExpertFFN)]\n    opt.step()\n    return {\'batch_task_ce\':ce.item(),\'balance_loss\':balance.item(),\'objective\':objective.item(),\n            \'gradient_norm_before_clip\':norm.item(),\'router_gradient_norm\':router_grad}\n\n\n@torch.no_grad()\ndef evaluate(model,batch,device):\n    model.eval()\n    loss=0.\n    loads=[]\n    entropy=[]\n    for x,y in zip(batch[0].split(8),batch[1].split(8)):\n        logits,aux=model(x.to(device))\n        loss+=F.cross_entropy(logits.reshape(-1,logits.shape[-1]),y.to(device).reshape(-1),reduction=\'sum\').item()\n        if aux:\n            counts=torch.stack([a[\'counts\'] for a in aux]).cpu()\n            if not loads:\n                loads=[torch.zeros_like(counts)]\n                entropy=[torch.zeros(len(aux))]\n            loads[0]+=counts\n            entropy[0]+=torch.tensor([a[\'entropy\'].item() for a in aux])*y.numel()\n    result={\'task_ce\':loss/batch[1].numel()}\n    if loads:\n        fractions=loads[0]/loads[0].sum(-1,keepdim=True)\n        result.update(expert_fractions=fractions.tolist(),\n                      router_entropy=(entropy[0]/batch[1].numel()).tolist(),\n                      expert_load_cv=(fractions.std(-1,unbiased=False)/fractions.mean(-1)).tolist())\n    return result\n\n\ndef phase(model,cfg,data,evals,seed,name,steps,lr,offset,device,checkpoint_dir,log_path):\n    opt=torch.optim.AdamW(model.parameters(),lr=lr,betas=(.9,.95),weight_decay=cfg[\'weight_decay\'])\n    gen=torch.Generator().manual_seed(seed+(10000 if name==\'dense_pretrain\' else 20000))\n    start=time.perf_counter()\n    records=[]\n    last={}\n    for i in range(steps+1):\n        if i%cfg[\'eval_every\']==0 or i==steps:\n            train=evaluate(model,evals[\'train\'],device)\n            validation=evaluate(model,evals[\'validation\'],device)\n            record={\'seed\':seed,\'phase\':name,\'step\':i,\'global_step\':offset+i,\n                    \'targets_seen\':(offset+i)*cfg[\'batch_size\']*cfg[\'model\'][\'context\'],\n                    \'train_ce\':train[\'task_ce\'],\'validation_ce\':validation[\'task_ce\'],\n                    \'elapsed_seconds_including_eval\':time.perf_counter()-start,**last}\n            for k in (\'expert_fractions\',\'router_entropy\',\'expert_load_cv\'):\n                if k in validation:\n                    record[k]=validation[k]\n            records.append(record)\n            with log_path.open(\'a\') as f:\n                f.write(json.dumps(record)+\'\\n\')\n            print(json.dumps(record),flush=True)\n        if i==steps:\n            break\n        last=step(model,opt,sample(data,gen,cfg[\'batch_size\'],cfg[\'model\'][\'context\']),device,cfg)\n    if device.startswith(\'cuda\'):\n        torch.cuda.synchronize()\n    elapsed=time.perf_counter()-start\n    saved={\'model\':model.state_dict(),\'optimizer\':opt.state_dict(),\'sampler_rng\':gen.get_state(),\n           \'torch_rng\':torch.get_rng_state(),\'cuda_rng\':torch.cuda.get_rng_state_all() if device.startswith(\'cuda\') else [],\n           \'config\':cfg,\'seed\':seed,\'phase\':name,\'step\':steps}\n    path=checkpoint_dir/f\'{name}.pt\'\n    torch.save(saved,path)\n    # Check on-disk model/Adam/sampler restoration with an identical next update.\n    loaded=torch.load(path,map_location=device,weights_only=True)\n    restored=copy.deepcopy(model)\n    restored.load_state_dict(loaded[\'model\'])\n    restored_opt=torch.optim.AdamW(restored.parameters(),lr=lr,betas=(.9,.95),weight_decay=cfg[\'weight_decay\'])\n    restored_opt.load_state_dict(loaded[\'optimizer\'])\n    restored_gen=torch.Generator()\n    restored_gen.set_state(loaded[\'sampler_rng\'].cpu())\n    batch=sample(data,gen,cfg[\'batch_size\'],cfg[\'model\'][\'context\'])\n    restored_batch=sample(data,restored_gen,cfg[\'batch_size\'],cfg[\'model\'][\'context\'])\n    assert all(torch.equal(a,b) for a,b in zip(batch,restored_batch))\n    reference=copy.deepcopy(model)\n    ref_opt=torch.optim.AdamW(reference.parameters(),lr=lr,betas=(.9,.95),weight_decay=cfg[\'weight_decay\'])\n    ref_opt.load_state_dict(copy.deepcopy(opt.state_dict()))\n    step(reference,ref_opt,batch,device,cfg)\n    step(restored,restored_opt,restored_batch,device,cfg)\n    error=max((a-b).abs().max().item() for a,b in zip(reference.parameters(),restored.parameters()))\n    assert error<1e-6,error\n    return records,{\'wall_seconds_including_eval\':elapsed,\n                    \'targets_per_second_including_eval\':steps*cfg[\'batch_size\']*cfg[\'model\'][\'context\']/elapsed,\n                    \'resume_next_update_parameter_error\':error}\n\n\n@torch.no_grad()\ndef generate(model,vocabulary,prompt,device,seed,length=200):\n    gen=torch.Generator(device=device).manual_seed(seed)\n    mapping={c:i for i,c in enumerate(vocabulary)}\n    tokens=torch.tensor([[mapping[c] for c in prompt]],device=device)\n    model.eval()\n    for _ in range(length):\n        logits,_=model(tokens[:,-model.cfg.context:])\n        next_token=torch.multinomial((logits[:,-1]/.8).softmax(-1),1,generator=gen)\n        tokens=torch.cat((tokens,next_token),dim=1)\n    return \'\'.join(vocabulary[i] for i in tokens[0].tolist())\n\n\ndef main():\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--config\',default=\'configs/gpt_cpu.json\')\n    p.add_argument(\'--data\',default=\'data/tinyshakespeare.txt\')\n    p.add_argument(\'--device\',default=\'cpu\')\n    p.add_argument(\'--output\',default=\'results/gpt-cpu-pilot\')\n    p.add_argument(\'--threads\',type=int,default=2)\n    p.add_argument(\'--resume-dense-run\',help=\'Prior result directory with completed dense phase and checkpoints/<run-name>/<seed>/dense_pretrain.pt\')\n    a=p.parse_args()\n    torch.set_num_threads(a.threads)\n    cfg=json.loads(Path(a.config).read_text())\n    out=Path(a.output)\n    out.mkdir(parents=True,exist_ok=False)\n    (out/\'config.json\').write_text(json.dumps(cfg,indent=2))\n    splits,manifest=load_data(Path(a.data),cfg[\'model\'][\'context\'])\n    manifest[\'objective\']=\'Causal next-character prediction at every position; input/target shifted by one.\'\n    (out/\'data_manifest.json\').write_text(json.dumps(manifest,indent=2))\n    evals={k:sample(v,torch.Generator().manual_seed(9000+i),cfg[\'eval_sequences\'],cfg[\'model\'][\'context\']) for i,(k,v) in enumerate(splits.items())}\n    try:\n        commit=subprocess.check_output([\'git\',\'rev-parse\',\'HEAD\'],stderr=subprocess.DEVNULL,text=True).strip()\n    except subprocess.CalledProcessError:\n        commit=None\n    hashes={str(p):hashlib.sha256(p.read_bytes()).hexdigest() for p in Path(\'src\').glob(\'*.py\')}\n    env={\'python\':platform.python_version(),\'torch\':str(torch.__version__),\'device\':a.device,\'threads\':a.threads,\n         \'gpu_name\':torch.cuda.get_device_name() if a.device.startswith(\'cuda\') else None,\n         \'git_commit\':commit,\'source_sha256\':hashes,\'precision\':\'FP32; no autocast/dropout\'}\n    (out/\'environment.json\').write_text(json.dumps(env,indent=2))\n    summaries=[]\n    samples={}\n    for seed in cfg[\'seeds\']:\n        torch.manual_seed(seed)\n        if a.device.startswith(\'cuda\'):\n            torch.cuda.reset_peak_memory_stats()\n        checkpoint_dir=Path(\'checkpoints\')/out.name/str(seed)\n        checkpoint_dir.mkdir(parents=True,exist_ok=True)\n        model=TinyGPT(GPTConfig(vocab_size=len(manifest[\'vocabulary\']),**cfg[\'model\'])).to(a.device)\n        if a.resume_dense_run:\n            source=Path(a.resume_dense_run)\n            source_manifest=json.loads((source/\'data_manifest.json\').read_text())\n            if source_manifest[\'sha256\']!=manifest[\'sha256\']:\n                raise ValueError(\'Resume dataset checksum mismatch\')\n            path=Path(\'checkpoints\')/source.name/str(seed)/\'dense_pretrain.pt\'\n            loaded=torch.load(path,map_location=a.device,weights_only=True)\n            if loaded[\'seed\']!=seed or loaded[\'phase\']!=\'dense_pretrain\' or loaded[\'step\']!=cfg[\'dense_steps\']:\n                raise ValueError(\'Resume checkpoint seed/phase/step mismatch\')\n            for key in (\'model\',\'dense_steps\',\'dense_lr\',\'batch_size\',\'weight_decay\',\'gradient_clip\',\'eval_sequences\'):\n                if loaded[\'config\'][key]!=cfg[key]:\n                    raise ValueError(f\'Resume configuration mismatch: {key}\')\n            model.load_state_dict(loaded[\'model\'])\n            torch.set_rng_state(loaded[\'torch_rng\'].cpu())\n            if a.device.startswith(\'cuda\') and loaded[\'cuda_rng\']:\n                torch.cuda.set_rng_state_all([r.cpu() for r in loaded[\'cuda_rng\']])\n            dense_records=[json.loads(line) for line in (source/\'metrics.jsonl\').read_text().splitlines()]\n            dense_records=[r for r in dense_records if r[\'seed\']==seed and r[\'phase\']==\'dense_pretrain\']\n            if not dense_records or dense_records[0][\'step\']!=0 or dense_records[-1][\'step\']!=cfg[\'dense_steps\']:\n                raise ValueError(\'Resume requires complete dense metrics\')\n            with (out/\'metrics.jsonl\').open(\'a\') as f:\n                for r in dense_records:f.write(json.dumps(r)+\'\\n\')\n            dense_timing={\'wall_seconds_including_eval\':dense_records[-1][\'elapsed_seconds_including_eval\'],\n                          \'resumed_from\':str(source),\'checkpoint_sha256\':hashlib.sha256(path.read_bytes()).hexdigest(),\n                          \'resume_next_update_parameter_error\':None,\n                          \'note\':\'Dense timing/metrics inherited; continuation optimizer resets as before.\'}\n            torch.save(loaded,checkpoint_dir/\'dense_pretrain.pt\')\n            print(f\'Restored dense checkpoint: {path}; skipping pretraining\',flush=True)\n        else:\n            dense_records,dense_timing=phase(model,cfg,splits[\'train\'],evals,seed,\'dense_pretrain\',cfg[\'dense_steps\'],cfg[\'dense_lr\'],0,a.device,checkpoint_dir,out/\'metrics.jsonl\')\n        model.eval()\n        moe=model.upcycle().eval()\n        # Exact parameter-copy checks catch structural errors independently of\n        # floating-point kernel differences in sparse subset execution.\n        for dense_block,moe_block in zip(model.blocks,moe.blocks):\n            for expert in moe_block.ffn.experts:\n                for original,copied in zip(dense_block.ffn.parameters(),expert.parameters()):\n                    if not torch.equal(original,copied):\n                        raise RuntimeError(\'Expert weight copy mismatch\')\n        with torch.no_grad():\n            probe,targets=(t.to(a.device) for t in evals[\'validation\'])\n            conversion=verify_model_conversion(model,moe,probe,targets)\n        conversion_error=conversion[\'max_absolute_logit_error\']\n        (out/f\'conversion-{seed}.json\').write_text(json.dumps(conversion,indent=2))\n        print(\'Conversion diagnostics:\',json.dumps(conversion),flush=True)\n        if not conversion[\'passed\']:\n            raise RuntimeError(f\'Conversion check failed; see {out}/conversion-{seed}.json: {conversion}\')\n        torch.save({\'model\':moe.state_dict(),\'config\':cfg,\'seed\':seed},checkpoint_dir/\'moe_converted.pt\')\n        original_experts={k:v.detach().clone() for k,v in moe.state_dict().items() if \'.experts.\' in k}\n        branches={}\n        for name,branch in [(\'dense_control\',copy.deepcopy(model)),(\'moe_continue\',moe)]:\n            records,timing=phase(branch,cfg,splits[\'train\'],evals,seed,name,cfg[\'continuation_steps\'],cfg[\'continuation_lr\'],cfg[\'dense_steps\'],a.device,checkpoint_dir,out/\'metrics.jsonl\')\n            branches[name]={\'start\':records[0],\'final\':records[-1],\'test_ce\':evaluate(branch,evals[\'test\'],a.device)[\'task_ce\'],**timing}\n            samples[f\'{seed}/{name}\']=generate(branch,manifest[\'vocabulary\'],\'First Citizen:\\n\',a.device,1234)\n        deltas={k:(v-original_experts[k]).norm().item() for k,v in moe.state_dict().items() if k in original_experts}\n        all_updated=all(v>0 for v in deltas.values())\n        acceptance={\'dense_train_drops\':dense_records[-1][\'train_ce\']<dense_records[0][\'train_ce\'],\n                    \'dense_validation_drops\':dense_records[-1][\'validation_ce\']<dense_records[0][\'validation_ce\'],\n                    \'moe_train_drops\':branches[\'moe_continue\'][\'final\'][\'train_ce\']<dense_records[-1][\'train_ce\'],\n                    \'moe_validation_drops\':branches[\'moe_continue\'][\'final\'][\'validation_ce\']<dense_records[-1][\'validation_ce\'],\n                    \'all_expert_matrices_updated\':all_updated,\'conversion_preserved\':conversion[\'passed\']}\n        summary={\'seed\':seed,\'model_config\':asdict(model.cfg),\'dense_parameters\':model.parameter_counts(),\n                 \'moe_parameters\':moe.parameter_counts(),\'conversion_max_logit_error\':conversion_error,\'conversion_diagnostics\':conversion,\n                 \'initial\':dense_records[0],\'dense_end\':dense_records[-1],\'dense_timing\':dense_timing,\n                 \'branches\':branches,\'expert_matrix_changes_l2\':deltas,\'acceptance\':acceptance,\n                 \'peak_cuda_memory_allocated_bytes\':torch.cuda.max_memory_allocated() if a.device.startswith(\'cuda\') else None}\n        summaries.append(summary)\n        (out/\'summary.json\').write_text(json.dumps(summaries,indent=2))\n        (out/\'generation_samples.json\').write_text(json.dumps(samples,indent=2))\n    print(\'Acceptance:\',json.dumps([s[\'acceptance\'] for s in summaries]),flush=True)\n    if not all(all(s[\'acceptance\'].values()) for s in summaries):\n        raise SystemExit(\'Acceptance failure; preserve the run and investigate, without overwriting it.\')\n\nif __name__==\'__main__\':\n    main()\n', 'src/train.py': '"""Run from repository root: python -m src.train --device cpu"""\nimport argparse\nimport copy\nimport csv\nimport hashlib\nimport json\nimport platform\nimport subprocess\nimport time\nfrom pathlib import Path\nimport torch\nfrom torch.nn import functional as F\nfrom .models import LinearLM, SparseMoE, parameters\n\n\ndef windows(data, starts, context):\n    return data[starts[:, None] + torch.arange(context)], data[starts + context]\n\n\ndef load_data(path, context):\n    raw = path.read_bytes()\n    text = raw.decode(\'utf-8\')\n    # Split raw text first: no input or target can cross a split boundary.\n    a, b = int(.8 * len(text)), int(.9 * len(text))\n    vocab = sorted(set(text[:a]))\n    mapping = {c: i for i, c in enumerate(vocab)}\n    # This corpus has the same character support in all splits; fail if changed.\n    splits = {k: torch.tensor([mapping[c] for c in t]) for k, t in\n              [(\'train\', text[:a]), (\'validation\', text[a:b]), (\'test\', text[b:])]}\n    manifest = {\'source\': \'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt\',\n                \'sha256\': hashlib.sha256(raw).hexdigest(), \'bytes\': len(raw),\n                \'characters\': len(text), \'vocabulary\': vocab,\n                \'split_characters\': {k: len(v) for k, v in splits.items()},\n                \'split_method\': \'contiguous 80/10/10; split before constructing windows\',\n                \'context\': context}\n    return splits, manifest\n\n\ndef fixed_batch(data, context, n, seed):\n    gen = torch.Generator().manual_seed(seed)\n    starts = torch.randint(len(data) - context, (n,), generator=gen)\n    return windows(data, starts, context)\n\n\n@torch.no_grad()\ndef evaluate(model, batch, device):\n    model.eval()\n    losses, counts, entropies = [], [], []\n    for x, y in zip(batch[0].split(512), batch[1].split(512)):\n        logits, aux = model(x.to(device))\n        losses.append(F.cross_entropy(logits, y.to(device), reduction=\'sum\').item())\n        if aux is not None:\n            counts.append(aux[\'counts\'].cpu())\n            entropies.append((aux[\'entropy\'].item(), len(y)))\n    out = {\'ce\': sum(losses) / len(batch[1])}\n    if counts:\n        load = torch.stack(counts).sum(0)\n        out[\'expert_fraction\'] = (load / load.sum()).tolist()\n        out[\'router_entropy\'] = sum(e*n for e,n in entropies) / len(batch[1])\n    return out\n\n\ndef update(model, optimizer, batch, device, cfg):\n    model.train()\n    optimizer.zero_grad(set_to_none=True)\n    logits, aux = model(batch[0].to(device))\n    ce = F.cross_entropy(logits, batch[1].to(device))\n    objective = ce + (cfg[\'balance_coefficient\'] * aux[\'balance\'] if aux else 0)\n    objective.backward()\n    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg[\'gradient_clip\'])\n    if not torch.isfinite(norm) or not torch.isfinite(objective):\n        raise RuntimeError(\'Non-finite gradient or loss\')\n    optimizer.step()\n    return {\'batch_ce\': ce.item(), \'objective\': objective.item(), \'grad_norm\': norm.item()}\n\n\ndef run_phase(model, cfg, splits, evals, device, seed, phase, steps, lr, offset, folder):\n    # Both continuation branches deliberately reset AdamW at conversion.\n    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=cfg[\'weight_decay\'])\n    gen = torch.Generator().manual_seed(seed + (10000 if phase == \'dense_pretrain\' else 20000))\n    records = []\n    start = time.perf_counter()\n    last = {}\n    for step in range(steps + 1):\n        if step % cfg[\'eval_every\'] == 0 or step == steps:\n            tr = evaluate(model, evals[\'train\'], device)\n            va = evaluate(model, evals[\'validation\'], device)\n            rec = {\'seed\': seed, \'phase\': phase, \'step\': step, \'global_step\': offset + step,\n                   \'targets_seen\': (offset + step) * cfg[\'batch_size\'],\n                   \'train_ce\': tr[\'ce\'], \'validation_ce\': va[\'ce\'],\n                   \'elapsed_seconds_including_eval\': time.perf_counter() - start,\n                   **last}\n            if \'expert_fraction\' in va:\n                rec.update(expert_fraction=va[\'expert_fraction\'], router_entropy=va[\'router_entropy\'])\n            records.append(rec)\n            print(json.dumps(rec), flush=True)\n        if step == steps:\n            break\n        indices = torch.randint(len(splits[\'train\']) - cfg[\'context\'], (cfg[\'batch_size\'],), generator=gen)\n        last = update(model, opt, windows(splits[\'train\'], indices, cfg[\'context\']), device, cfg)\n    if device.startswith(\'cuda\'):\n        torch.cuda.synchronize()\n    elapsed = time.perf_counter() - start\n    checkpoint = {\'model\': model.state_dict(), \'optimizer\': opt.state_dict(),\n                  \'sampler_rng\': gen.get_state(), \'torch_rng\': torch.get_rng_state(),\n                  \'cuda_rng\': torch.cuda.get_rng_state_all() if device.startswith(\'cuda\') else [],\n                  \'phase\': phase, \'step\': steps, \'seed\': seed, \'config\': cfg}\n    checkpoint_path = folder / f\'{phase}.pt\'\n    torch.save(checkpoint, checkpoint_path)\n    # Verify actual disk reload, optimizer restoration, sampler restoration and next update.\n    loaded = torch.load(checkpoint_path, map_location=device, weights_only=True)\n    clone = copy.deepcopy(model)\n    clone.load_state_dict(loaded[\'model\'])\n    clone_opt = torch.optim.AdamW(clone.parameters(), lr=lr, weight_decay=cfg[\'weight_decay\'])\n    clone_opt.load_state_dict(loaded[\'optimizer\'])\n    clone_gen = torch.Generator()\n    clone_gen.set_state(loaded[\'sampler_rng\'].cpu())\n    next_a = torch.randint(len(splits[\'train\']) - cfg[\'context\'], (cfg[\'batch_size\'],), generator=gen)\n    next_b = torch.randint(len(splits[\'train\']) - cfg[\'context\'], (cfg[\'batch_size\'],), generator=clone_gen)\n    assert torch.equal(next_a, next_b)\n    # Work on a copy so verification cannot alter the measured final model.\n    reference = copy.deepcopy(model)\n    ref_opt = torch.optim.AdamW(reference.parameters(), lr=lr, weight_decay=cfg[\'weight_decay\'])\n    ref_opt.load_state_dict(copy.deepcopy(opt.state_dict()))\n    update(reference, ref_opt, windows(splits[\'train\'], next_a, cfg[\'context\']), device, cfg)\n    update(clone, clone_opt, windows(splits[\'train\'], next_b, cfg[\'context\']), device, cfg)\n    resume_error = max((x-y).abs().max().item() for x,y in zip(reference.parameters(), clone.parameters()))\n    assert resume_error < 1e-6, resume_error\n    return records, {\'seconds_including_eval_and_save_excluded\': elapsed,\n                     \'target_tokens_per_second_including_eval\': steps * cfg[\'batch_size\'] / elapsed,\n                     \'resume_next_step_max_parameter_error\': resume_error}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--config\', default=\'configs/routing.json\')\n    parser.add_argument(\'--data\', default=\'data/tinyshakespeare.txt\')\n    parser.add_argument(\'--device\', default=\'cpu\')\n    parser.add_argument(\'--output\', default=\'results/routing\')\n    parser.add_argument(\'--seeds\', type=int, nargs=\'+\')\n    parser.add_argument(\'--threads\', type=int, default=2)\n    args = parser.parse_args()\n    torch.set_num_threads(args.threads)\n    cfg = json.loads(Path(args.config).read_text())\n    if args.seeds:\n        cfg[\'seeds\'] = args.seeds\n    out = Path(args.output)\n    out.mkdir(parents=True, exist_ok=False)  # never overwrite previous evidence\n    cfg_path = out / \'config.json\'\n    cfg_path.write_text(json.dumps(cfg, indent=2))\n    splits, manifest = load_data(Path(args.data), cfg[\'context\'])\n    (out / \'data_manifest.json\').write_text(json.dumps(manifest, indent=2))\n    evals = {k: fixed_batch(v, cfg[\'context\'], cfg[\'eval_samples\'], 9000+i) for i,(k,v) in enumerate(splits.items())}\n    try:\n        commit = subprocess.check_output([\'git\',\'rev-parse\',\'HEAD\'], stderr=subprocess.DEVNULL, text=True).strip()\n    except subprocess.CalledProcessError:\n        commit = None\n    source_hashes = {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(Path(\'src\').glob(\'*.py\'))}\n    environment = {\'python\': platform.python_version(), \'torch\': str(torch.__version__),\n                   \'platform\': platform.platform(), \'device\': args.device, \'threads\': args.threads,\n                   \'git_commit\': commit, \'source_sha256\': source_hashes,\n                   \'gpu_name\': torch.cuda.get_device_name() if args.device.startswith(\'cuda\') else None}\n    (out / \'environment.json\').write_text(json.dumps(environment, indent=2))\n    all_records, summaries = [], []\n    for seed in cfg[\'seeds\']:\n        torch.manual_seed(seed)\n        checkpoint_dir = Path(\'checkpoints\') / out.name / str(seed)\n        checkpoint_dir.mkdir(parents=True, exist_ok=True)\n        dense = LinearLM(cfg[\'context\'], len(manifest[\'vocabulary\'])).to(args.device)\n        records, timing = run_phase(dense,cfg,splits,evals,args.device,seed,\'dense_pretrain\',\n                                   cfg[\'dense_steps\'],cfg[\'dense_lr\'],0,checkpoint_dir)\n        all_records += records\n        initial, dense_end = records[0], records[-1]\n        moe = SparseMoE(dense, cfg[\'experts\'], cfg[\'top_k\']).to(args.device)\n        with torch.no_grad():\n            probe = evals[\'validation\'][0].to(args.device)\n            max_error = (dense(probe)[0] - moe(probe)[0]).abs().max().item()\n        assert max_error < 1e-5, max_error\n        torch.save({\'model\':moe.state_dict(),\'config\':cfg,\'seed\':seed},checkpoint_dir/\'moe_converted.pt\')\n        moe_initial_state = copy.deepcopy(moe.state_dict())\n        branches = {}\n        for name, model in [(\'dense_control\', copy.deepcopy(dense)), (\'moe_continue\', moe)]:\n            recs, stats = run_phase(model,cfg,splits,evals,args.device,seed,name,\n                                   cfg[\'continuation_steps\'],cfg[\'continuation_lr\'],cfg[\'dense_steps\'],checkpoint_dir)\n            all_records += recs\n            # Test is measured only at final checkpoints, never used for model selection.\n            branches[name] = {\'start\':recs[0], \'final\':recs[-1], \'test_ce\':evaluate(model,evals[\'test\'],args.device)[\'ce\'], **stats}\n        expert_deltas = [(expert.weight - moe_initial_state[f\'experts.{i}.weight\']).norm().item()\n                         for i,expert in enumerate(moe.experts)]\n        summary = {\'seed\':seed,\'dense_parameters\':parameters(dense),\'moe_total_parameters\':parameters(moe),\n                   \'moe_active_parameters_per_target\':parameters(moe.router)+cfg[\'top_k\']*parameters(dense.linear),\n                   \'conversion_max_logit_error\':max_error,\'initial\':initial,\'dense_end\':dense_end,\n                   \'dense_pretrain_timing\':timing, \'branches\':branches, \'expert_weight_change_l2\':expert_deltas,\n                   \'expert_pair_distance_l2\':(moe.experts[0].weight-moe.experts[1].weight).norm().item()}\n        summary[\'acceptance\'] = {\'dense_train_loss_drops\':dense_end[\'train_ce\'] < initial[\'train_ce\'],\n                                \'dense_validation_loss_drops\':dense_end[\'validation_ce\'] < initial[\'validation_ce\'],\n                                \'moe_train_loss_drops\':branches[\'moe_continue\'][\'final\'][\'train_ce\'] < dense_end[\'train_ce\'],\n                                \'moe_validation_loss_drops\':branches[\'moe_continue\'][\'final\'][\'validation_ce\'] < dense_end[\'validation_ce\'],\n                                \'all_experts_updated\':all(x>0 for x in expert_deltas),\n                                \'conversion_preserved\':max_error<1e-5}\n        summaries.append(summary)\n        (out / \'summary.json\').write_text(json.dumps(summaries, indent=2))\n        with (out / \'metrics.jsonl\').open(\'w\') as f:\n            for rec in all_records:\n                f.write(json.dumps(rec)+\'\\n\')\n    keys = [\'seed\',\'phase\',\'step\',\'global_step\',\'targets_seen\',\'train_ce\',\'validation_ce\',\'elapsed_seconds_including_eval\',\'batch_ce\',\'objective\',\'grad_norm\',\'router_entropy\']\n    with (out / \'metrics.csv\').open(\'w\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=keys,extrasaction=\'ignore\'); writer.writeheader(); writer.writerows(all_records)\n    print(\'Acceptance:\', json.dumps([s[\'acceptance\'] for s in summaries]))\n    if not all(all(s[\'acceptance\'].values()) for s in summaries):\n        raise SystemExit(\'One or more acceptance checks failed; preserve evidence and investigate.\')\n\nif __name__ == \'__main__\':\n    main()\n', 'src/models.py': '"""Literal affine predictor and sparse, function-preserving MoE conversion."""\nimport copy\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nclass LinearLM(nn.Module):\n    def __init__(self, context, vocab):\n        super().__init__()\n        self.context, self.vocab = context, vocab\n        self.linear = nn.Linear(context * vocab, vocab)\n\n    def features(self, tokens):\n        return F.one_hot(tokens, self.vocab).flatten(1).float()\n\n    def forward(self, tokens):\n        return self.linear(self.features(tokens)), None\n\nclass SparseMoE(nn.Module):\n    def __init__(self, dense, experts=4, top_k=2):\n        super().__init__()\n        if not 1 <= top_k <= experts:\n            raise ValueError(\'Require 1 <= top_k <= experts\')\n        self.context, self.vocab = dense.context, dense.vocab\n        self.top_k = top_k\n        self.experts = nn.ModuleList([copy.deepcopy(dense.linear) for _ in range(experts)])\n        self.router = nn.Linear(self.context * self.vocab, experts)\n        nn.init.normal_(self.router.weight, std=0.02)\n        nn.init.zeros_(self.router.bias)\n\n    def forward(self, tokens):\n        x = F.one_hot(tokens, self.vocab).flatten(1).float()\n        router_logits = self.router(x)\n        probs = router_logits.softmax(-1)\n        selected_logits, ids = router_logits.topk(self.top_k, dim=-1)\n        weights = selected_logits.softmax(-1)\n        output = x.new_zeros(x.shape[0], self.vocab)\n        # Actual sparse dispatch: each expert sees only its assigned samples.\n        for i, expert in enumerate(self.experts):\n            rows, slots = torch.where(ids == i)\n            if rows.numel():\n                contribution = expert(x[rows]) * weights[rows, slots, None]\n                output = output.index_add(0, rows, contribution)\n        counts = torch.bincount(ids.flatten(), minlength=len(self.experts)).float()\n        fraction = counts / ids.numel()\n        balance = len(self.experts) * (fraction.detach() * probs.mean(0)).sum()\n        entropy = -(probs * probs.clamp_min(1e-9).log()).sum(-1).mean()\n        return output, {\'balance\': balance, \'counts\': counts, \'entropy\': entropy}\n\ndef parameters(model):\n    return sum(p.numel() for p in model.parameters())\n', 'scripts/prepare_tinystories_lab.py': '"""Prepare a frozen bounded TinyStories subset and train a BPE on TRAIN stories only."""\nimport argparse,hashlib,json\nfrom pathlib import Path\nimport numpy as np\nfrom datasets import load_dataset\nfrom huggingface_hub import HfApi\nfrom tokenizers import Tokenizer,models,trainers,pre_tokenizers,decoders\n\np=argparse.ArgumentParser()\np.add_argument(\'--output\',default=\'data/tinystories_lab\')\np.add_argument(\'--train-stories\',type=int,default=12000)\np.add_argument(\'--heldout-stories\',type=int,default=1200)\np.add_argument(\'--vocab-size\',type=int,default=4096)\np.add_argument(\'--revision\',default=\'f54c09fd23315a6f9c86f9dc80f725de7d8f9c64\')\np.add_argument(\'--tokenizer\',help=\'Reuse the archived tokenizer for exact token IDs instead of fitting a new one\')\na=p.parse_args()\nroot=Path(a.output);root.mkdir(parents=True,exist_ok=False)\nrevision=a.revision\nsplits={}\nfor key,n in [(\'train\',a.train_stories),(\'validation\',a.heldout_stories*2)]:\n ds=load_dataset(\'roneneldan/TinyStories\',split=key,revision=revision,streaming=True)\n # Bounded shuffled sample; order and revision are recorded, not a globally uniform dataset sample.\n ds=ds.shuffle(seed=17,buffer_size=2000)\n splits[key]=[r[\'text\'] for r in ds.take(n)]\ntrain=splits[\'train\']; held=splits[\'validation\']\n# Remove exact document overlaps across splits; near duplicates are not addressed here.\nseen=set(hashlib.sha256(t.encode()).hexdigest() for t in train)\nvalidation=[];test=[]\nfor i,text in enumerate(held):\n digest=hashlib.sha256(text.encode()).hexdigest()\n if digest in seen:continue\n seen.add(digest)\n (validation if i%2==0 else test).append(text)\nif a.tokenizer:\n tokenizer=Tokenizer.from_file(a.tokenizer)\n if tokenizer.get_vocab_size()!=a.vocab_size or tokenizer.token_to_id(\'[EOS]\') is None:\n  raise ValueError(\'Archived tokenizer vocabulary/special tokens do not match configuration\')\nelse:\n tokenizer=Tokenizer(models.BPE(unk_token=\'[UNK]\'))\n tokenizer.pre_tokenizer=pre_tokenizers.ByteLevel(add_prefix_space=False)\n tokenizer.decoder=decoders.ByteLevel()\n trainer=trainers.BpeTrainer(vocab_size=a.vocab_size,special_tokens=[\'[UNK]\',\'[EOS]\'],initial_alphabet=pre_tokenizers.ByteLevel.alphabet())\n tokenizer.train_from_iterator(train,trainer=trainer)\nif a.tokenizer:\n import shutil\n shutil.copyfile(a.tokenizer,root/\'tokenizer.json\')\nelse:\n tokenizer.save(str(root/\'tokenizer.json\'))\neos=tokenizer.token_to_id(\'[EOS]\')\nencoded={}\nfor key,stories in [(\'train\',train),(\'validation\',validation),(\'test\',test)]:\n ids=[]\n for text in stories:ids.extend(tokenizer.encode(text).ids+[eos])\n encoded[key]=np.asarray(ids,dtype=np.int32)\nnp.savez(root.with_suffix(\'.npz\'),**encoded,vocab_size=tokenizer.get_vocab_size())\nmanifest={\'dataset\':\'roneneldan/TinyStories\',\'revision\':revision,\'shuffle_seed\':17,\'shuffle_buffer\':2000,\n          \'train_stories\':len(train),\'validation_stories\':len(validation),\'test_stories\':len(test),\n          \'token_counts\':{k:len(v) for k,v in encoded.items()},\'vocab_size\':tokenizer.get_vocab_size(),\n          \'tokenizer_sha256\':hashlib.sha256((root/\'tokenizer.json\').read_bytes()).hexdigest(),\n          \'packed_policy\':\'concatenate documents with EOS; causal attention may cross EOS\',\n          \'deduplication\':\'exact cross-split document hashing only; not near-duplicate decontamination\'}\n(root/\'manifest.json\').write_text(json.dumps(manifest,indent=2))\nprint(json.dumps(manifest,indent=2))\n', 'configs/granular_t4.json': '{\n  "model": {"context":128,"layers":4,"width":192,"heads":6,"intermediate":384,"experts":4,"top_k":1},\n  "conversion": {"partitions":4,"replicas":4,"top_k":4},\n  "batch_size":4,"eval_sequences":32,"seeds":[17],\n  "dense_steps":500,"continuation_steps":500,"dense_lr":0.0006,"continuation_lr":0.0003,\n  "balance_coefficient":0.01,"weight_decay":0.01,"gradient_clip":1.0,"eval_every":50\n}\n', 'tests/test_granular.py': "import unittest\nimport torch\nfrom src.gpt import SwiGLU, TinyGPT, GPTConfig\nfrom src.granular import GranularFFN, partition_upcycle\n\n\nclass GranularTests(unittest.TestCase):\n    def setUp(self):\n        torch.manual_seed(23)\n        torch.set_num_threads(2)\n        self.dense = SwiGLU(16, 32)\n        self.x = torch.randn(3, 7, 16, requires_grad=True)\n\n    def test_all_partitions_reconstruct_and_copy_coordinates(self):\n        moe = GranularFFN(self.dense, 4, 2, 8)\n        with torch.no_grad():\n            moe.router.weight.zero_()\n        torch.testing.assert_close(moe(self.x)[0], self.dense(self.x))\n        for e, idx in zip(moe.experts, moe.unit_indices):\n            torch.testing.assert_close(e.gate.weight, self.dense.gate.weight[idx])\n            torch.testing.assert_close(e.up.weight, self.dense.up.weight[idx])\n            torch.testing.assert_close(e.down.weight, self.dense.down.weight[:, idx])\n\n    def test_sparse_reference_dispatch_and_gradients(self):\n        moe = GranularFFN(self.dense, 4, 3, 4)\n        flat = self.x.reshape(-1, 16)\n        values, ids = moe.router(flat).topk(4, -1)\n        ref = torch.stack([e(flat) for e in moe.experts], 1)\n        ref = (ref[torch.arange(len(flat))[:, None], ids] * values.softmax(-1)[:, :, None]).sum(1) * 4\n        # Hooks verify each expert receives precisely its selected rows.\n        observed = [0] * len(moe.experts)\n        handles = [e.register_forward_pre_hook(lambda m, args, i=i: observed.__setitem__(i, len(args[0])))\n                   for i, e in enumerate(moe.experts)]\n        actual, aux = moe(self.x)\n        for h in handles:\n            h.remove()\n        torch.testing.assert_close(actual.reshape(-1, 16), ref)\n        self.assertEqual(observed, aux['counts'].int().tolist())\n        self.assertEqual(sum(observed), len(flat) * 4)\n        actual.square().mean().backward()\n        self.assertGreater(moe.router.weight.grad.norm().item(), 0)\n        self.assertTrue(all(torch.isfinite(p.grad).all() for p in moe.parameters() if p.grad is not None))\n\n    def test_model_dimensions_accounting_and_invalid_shapes(self):\n        model = TinyGPT(GPTConfig(vocab_size=13, context=8, layers=1, width=16, heads=4, intermediate=32))\n        moe = partition_upcycle(model, 4, 3, 4)\n        each = sum(p.numel() for p in moe.blocks[0].ffn.experts[0].parameters())\n        original = sum(p.numel() for p in model.blocks[0].ffn.parameters())\n        self.assertEqual(each * 4, original)\n        self.assertEqual(each * 12, original * 3)\n        self.assertEqual(moe.cfg.intermediate, 8)\n        self.assertEqual(model.cfg.intermediate, 32)\n        self.assertEqual(moe(torch.randint(13, (2, 8)))[0].shape, (2, 8, 13))\n        with self.assertRaises(ValueError):\n            GranularFFN(self.dense, 3, 2, 2)\n", 'tests/test_gpt.py': "import unittest\nimport torch\nfrom src.gpt import GPTConfig, TinyGPT, ExpertFFN\n\nclass GPTTests(unittest.TestCase):\n    def setUp(self):\n        torch.manual_seed(17)\n        torch.set_num_threads(2)\n        self.model=TinyGPT(GPTConfig(vocab_size=13,context=12,layers=2,width=16,heads=4,intermediate=32))\n        self.tokens=torch.randint(13,(3,12))\n\n    def test_conversion_preserves_full_gpt_logits(self):\n        converted=self.model.upcycle()\n        torch.testing.assert_close(self.model(self.tokens)[0],converted(self.tokens)[0],atol=1e-6,rtol=1e-5)\n        self.assertEqual(len(converted(self.tokens)[1]),2)\n        self.assertIs(converted.head.weight,converted.token_embedding.weight)\n\n    def test_future_token_does_not_change_past_logits(self):\n        for model in (self.model,self.model.upcycle()):\n            changed=self.tokens.clone()\n            changed[:,7:] = (changed[:,7:]+1)%13\n            torch.testing.assert_close(model(self.tokens)[0][:,:7],model(changed)[0][:,:7])\n\n    def test_sparse_ffn_reference_and_gradients(self):\n        moe=self.model.upcycle().blocks[0].ffn\n        with torch.no_grad():\n            moe.experts[0].down.weight.add_(.1)\n        x=torch.randn(3,12,16,requires_grad=True)\n        flat=x.reshape(-1,16)\n        values,ids=moe.router(flat).topk(2,dim=-1)\n        all_outputs=torch.stack([e(flat) for e in moe.experts],dim=1)\n        ref=(all_outputs[torch.arange(len(flat))[:,None],ids]*values.softmax(-1)[:,:,None]).sum(1)\n        actual,aux=moe(x)\n        torch.testing.assert_close(actual.reshape(-1,16),ref)\n        self.assertEqual(int(aux['counts'].sum()),2*len(flat))\n        actual.square().mean().backward()\n        self.assertGreater(moe.router.weight.grad.norm().item(),0)\n\n    def test_parameter_accounting(self):\n        counts=self.model.upcycle().parameter_counts()\n        self.assertGreater(counts['total'],counts['active_parameter_convention'])\n        self.assertGreater(counts['active_parameter_convention'],self.model.parameter_counts()['total'])\n\nif __name__=='__main__':\n    unittest.main()\n", 'src/conversion_checks.py': '"""Scale-aware FP32 conversion checks with retained behavioral diagnostics."""\nimport copy\nimport torch\nfrom torch.nn import functional as F\n\n\n@torch.no_grad()\ndef conversion_diagnostics(reference, actual, targets, atol=1e-5, rtol=1e-5):\n    if reference.shape != actual.shape:\n        raise ValueError(\'Conversion changed output shape\')\n    difference = (actual-reference).abs()\n    allowed = atol + rtol*reference.abs()\n    finite = bool(torch.isfinite(reference).all() and torch.isfinite(actual).all())\n    dense_ce = F.cross_entropy(reference.reshape(-1,reference.shape[-1]),targets.reshape(-1)).item()\n    moe_ce = F.cross_entropy(actual.reshape(-1,actual.shape[-1]),targets.reshape(-1)).item()\n    close = bool(torch.all(difference <= allowed)) if finite else False\n    ce_close = abs(dense_ce-moe_ce) <= 1e-5\n    return {\'atol\':atol,\'rtol\':rtol,\'ce_atol\':1e-5,\'finite\':finite,\n            \'max_absolute_logit_error\':difference.max().item(),\n            \'rms_logit_error\':difference.square().mean().sqrt().item(),\n            \'max_error_over_allowed\':(difference/allowed).max().item(),\n            \'logits_outside_tolerance\':int((difference>allowed).sum().item()),\n            \'dense_probe_ce\':dense_ce,\'moe_probe_ce\':moe_ce,\'probe_ce_change\':moe_ce-dense_ce,\n            \'max_probability_error\':(actual.softmax(-1)-reference.softmax(-1)).abs().max().item(),\n            \'logits_close\':close,\'ce_close\':ce_close,\'passed\':finite and close and ce_close}\n\n\n@torch.no_grad()\ndef verify_model_conversion(dense, moe, tokens, targets):\n    """Validate full-copy conversion in FP32 and independent FP64 CPU probes.\n\n    Retain the original per-logit FP32 check. If it fails, accept only when\n    a tighter CPU-double reference passes AND explicit FP32 behavioral and\n    error bounds pass. This is not a generic license to ignore mismatches.\n    """\n    stats = conversion_diagnostics(dense(tokens)[0], moe(tokens)[0], targets)\n    indices = torch.linspace(0, len(tokens)-1, min(8, len(tokens))).long()\n    probe = tokens.detach().cpu()[indices]\n    labels = targets.detach().cpu()[indices]\n    dense64 = copy.deepcopy(dense).cpu().double().eval()\n    moe64 = copy.deepcopy(moe).cpu().double().eval()\n    ref = dense64(probe)[0]\n    actual = moe64(probe)[0]\n    difference = (actual-ref).abs()\n    finite64 = bool(torch.isfinite(ref).all() and torch.isfinite(actual).all())\n    close64 = bool(torch.all(difference <= 1e-10 + 1e-10*ref.abs())) if finite64 else False\n    ce_ref = F.cross_entropy(ref.reshape(-1,ref.shape[-1]),labels.reshape(-1)).item()\n    ce_actual = F.cross_entropy(actual.reshape(-1,actual.shape[-1]),labels.reshape(-1)).item()\n    reference_passed = finite64 and close64 and abs(ce_actual-ce_ref) <= 1e-10\n    stats[\'fp64_cpu_reference\'] = {\n        \'sequence_indices\':indices.tolist(),\'atol\':1e-10,\'rtol\':1e-10,\'ce_atol\':1e-10,\n        \'max_absolute_logit_error\':difference.max().item(),\n        \'probe_ce_change\':ce_actual-ce_ref,\'passed\':reference_passed}\n    bounds = {\'max_absolute_logit_error\':1e-4, \'rms_logit_error\':1e-5,\n              \'max_probability_error\':1e-5, \'absolute_probe_ce_change\':1e-5}\n    bounded = (stats[\'finite\'] and stats[\'max_absolute_logit_error\'] <= bounds[\'max_absolute_logit_error\']\n               and stats[\'rms_logit_error\'] <= bounds[\'rms_logit_error\']\n               and stats[\'max_probability_error\'] <= bounds[\'max_probability_error\']\n               and abs(stats[\'probe_ce_change\']) <= bounds[\'absolute_probe_ce_change\'])\n    original_passed = stats[\'passed\']\n    stats[\'fp32_behavior_bounds\'] = bounds\n    stats[\'fp32_behavior_bounds_passed\'] = bounded\n    stats[\'passed\'] = reference_passed and (original_passed or bounded)\n    stats[\'acceptance_method\'] = (\'fp32_per_logit_and_fp64_reference\' if original_passed else\n                                  \'fp64_reference_and_bounded_fp32\') if stats[\'passed\'] else \'rejected\'\n    return stats\n'}
for name, content in payload.items():
    path = root / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)


## Prepare data and freeze the run

Download 12,000 training stories and up to 2,400 held-out stories. The official validation split supplies our disjoint validation/test subsets. Record the dataset revision, exact cross-split deduplication, tokenizer checksum and token counts. Attention may cross document boundaries separated by EOS. Near-duplicate removal is outside this initial experiment.

The default run uses one seed, 500 dense steps and 500 steps for each of three continuations (256,000 target tokens per phase). FP32, batch 4, context 128, four layers and width 192 are conservative starting choices. The small unit tests use CPU. The training command below explicitly uses CUDA.


In [ ]:
if not Path("data/tinystories_lab.npz").exists():
    subprocess.run([sys.executable, "scripts/prepare_tinystories_lab.py"], check=True)
assert Path("data/tinystories_lab/manifest.json").exists(), "Incomplete data preparation; inspect before rerunning."
print(Path("data/tinystories_lab/manifest.json").read_text())
cfg = json.loads(Path("configs/granular_t4.json").read_text())
# Change these BEFORE training; the result archive freezes the chosen configuration.
# cfg["seeds"] = [17, 42, 123]  # Confirmation after inspecting the first run
print(json.dumps(cfg, indent=2))
Path("configs/run.json").write_text(json.dumps(cfg, indent=2))


## Train and continue

Each branch is trained separately on GPU. Checkpoint recovery verifies an identical next optimizer update. The phase-end check temporarily holds extra model/optimizer copies, so peak memory includes verification. Checkpoints and metrics remain on disk if execution fails; do not overwrite a result directory.

Task CE is reported separately from the auxiliary balancing objective. A successful continuation reduces CE from its **own conversion start**; recovering the dense checkpoint's CE is a second, stronger check. Do not select settings using final test CE.


In [ ]:
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
run_dir = "results/granular-" + stamp
subprocess.run([sys.executable, "-m", "src.granular_experiment",
                "--config", "configs/run.json", "--data", "data/tinystories_lab.npz",
                "--device", "cuda", "--output", run_dir], check=True)


## Inspect conversion, continued loss and expert utilization

Read conversion CE jumps, final validation CE, selected FFN width, parameter counts, memory, expert weight changes and checkpoint recovery before growing the model. The final probe's zero-allocation count describes that fixed validation batch; it does not establish persistent expert death.


In [ ]:
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
summary = json.loads((Path(run_dir) / "summary.json").read_text())
display(Markdown((Path(run_dir) / "REPORT.md").read_text()))
records = [json.loads(line) for line in (Path(run_dir) / "metrics.jsonl").read_text().splitlines()]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for seed in cfg["seeds"]:
    for arm in ("dense_pretrain", "dense_control", "full_copy", "granular"):
        rows = [r for r in records if r["seed"] == seed and r["phase"] == arm]
        axes[0].plot([r["global_step"] for r in rows], [r["validation_ce"] for r in rows], label=f"{seed}/{arm}")
for result in summary:
    for arm, b in result["branches"].items():
        print(result["seed"], arm, json.dumps({k: b[k] for k in (
            "parameters", "selected_ffn_width", "ffn_matrix_macs_per_token_per_layer",
            "router_matrix_macs_per_token_per_layer", "conversion_max_logit_drift", "conversion_ce_jump",
            "validation_drops_from_conversion", "validation_recovers_dense_checkpoint",
            "all_expert_matrices_updated", "resume_next_update_parameter_error", "peak_cuda_allocated_bytes")}, indent=2))
        if "expert_fractions" in b["final"]:
            for layer, fractions in enumerate(b["final"]["expert_fractions"]):
                axes[1].plot(range(len(fractions)), fractions, marker=".", label=f"{result['seed']}/{arm}/layer{layer}")
                print("Fixed-probe zero allocations:", arm, layer, sum(v == 0 for v in fractions))
axes[0].set(xlabel="Training step (same target tokens/step)", ylabel="Validation task CE")
axes[1].set(xlabel="Expert index", ylabel="Final fixed-probe routing fraction")
for ax in axes:
    ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(Path(run_dir) / "curves.png", dpi=160)
plt.show()


## Download evidence

Archive source, configurations, tokenizer, provenance, metrics, plots and checkpoints. Packed token arrays are excluded to keep the download smaller; the dataset revision and preparation policy are preserved. Download before Colab disconnects.


In [ ]:
import zipfile
from google.colab import files
archive = Path("/content/MoE_Granular_Lab-evidence.zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for folder in ("src", "scripts", "tests", "configs", "results", "checkpoints", "data/tinystories_lab"):
        for path in (root / folder).rglob("*"):
            if path.is_file() and "__pycache__" not in path.parts:
                z.write(path, path.relative_to(root))
print("Archive MiB:", archive.stat().st_size / 2**20)
files.download(str(archive))
